In [1]:
import gym
import numpy as np
import random
import matplotlib.pyplot as plt
import tensorflow as tf

from collections import deque

print("Gym version:", gym.__version__)


C:\Users\linsi\anaconda3\envs\gpu_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Gym version: 0.17.3


## 1.Training settings

In [2]:

GRAVITY_SETTINGS = {
    "Default Gravity": 10.0,
    "Free Fall": 0.0,
    "Anti-Gravity": -10.0,
    "Supergravity": 15.0
}

TRAINING_SEEDS = [11, 22, 33]

# Used when choosing the best hyperparameter configuration
VALIDATION_ENV_SEEDS = list(range(1000, 1010))

# Unseen seeds used only for final testing
TEST_ENV_SEEDS = list(range(2000, 2020))

# Shorter runs for hyperparameter screening
SEARCH_EPISODES = 150

# Longer runs for the final gravity experiments
FINAL_EPISODES = 350


# The pendulum is considered balanced when it is:
# 1. Within 12 degrees of the upright position
# 2. Moving at no more than 1 radian per second

UPRIGHT_ANGLE_LIMIT = np.deg2rad(12)
UPRIGHT_SPEED_LIMIT = 1.0

# Discretise continuous torque range [-2, 2]
NUM_ACTIONS = 9
TORQUE_VALUES = np.linspace(-2.0, 2.0, NUM_ACTIONS).astype(np.float32)

print("Discrete torque values:")
print(TORQUE_VALUES)

Discrete torque values:
[-2.  -1.5 -1.  -0.5  0.   0.5  1.   1.5  2. ]


In [3]:
def preprocess_state(state):
 
    processed_state = np.array(state, dtype=np.float32).copy()

    processed_state[2] = processed_state[2] / 8.0

    return processed_state

In [4]:
def create_pendulum_env(gravity=10.0):
   
    return gym.make(
        "Pendulum-v0",
        g=gravity
    )

In [5]:
# Test the default-gravity environment

env = create_pendulum_env(gravity=10.0)

env.seed(42)
observation = env.reset()

print("Initial observation:", observation)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Maximum episode steps:", env.spec.max_episode_steps)

env.close()

Initial observation: [ 0.70329069 -0.71090239 -0.0313229 ]
Observation space: Box(-8.0, 8.0, (3,), float32)
Action space: Box(-2.0, 2.0, (1,), float32)
Maximum episode steps: 200


In [6]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(
        self,
        state,
        action,
        reward,
        next_state,
        terminal
    ):
        self.buffer.append(
            (
                state,
                action,
                reward,
                next_state,
                terminal
            )
        )

    def sample(self, batch_size):
        batch = random.sample(
            self.buffer,
            batch_size
        )

        states = np.asarray(
            [transition[0] for transition in batch],
            dtype=np.float32
        )

        actions = np.asarray(
            [transition[1] for transition in batch],
            dtype=np.int32
        )

        rewards = np.asarray(
            [transition[2] for transition in batch],
            dtype=np.float32
        )

        next_states = np.asarray(
            [transition[3] for transition in batch],
            dtype=np.float32
        )

        terminals = np.asarray(
            [transition[4] for transition in batch],
            dtype=np.float32
        )

        return (
            states,
            actions,
            rewards,
            next_states,
            terminals
        )

    def __len__(self):
        return len(self.buffer)

In [7]:
class DoubleDQNAgent:
    def __init__(self, config):
        self.config = config.copy()

        self.state_dimension = 3
        self.number_of_actions = config["number_of_actions"]

        self.gamma = config["gamma"]
        self.batch_size = config["batch_size"]

        self.target_update_train_steps = (
            config["target_update_train_steps"]
        )

        self.torque_values = create_discrete_torques(
            self.number_of_actions
        )

        self.replay_buffer = ReplayBuffer(
            config["replay_capacity"]
        )

        self.online_model = self.build_model(
            hidden_layers=config["hidden_layers"],
            learning_rate=config["learning_rate"]
        )

        self.target_model = self.build_model(
            hidden_layers=config["hidden_layers"],
            learning_rate=config["learning_rate"]
        )

        # The target network initially has the same weights
        # as the online network.
        self.target_model.set_weights(
            self.online_model.get_weights()
        )

        self.optimizer = tf.keras.optimizers.Adam(
            learning_rate=config["learning_rate"]
        )

        self.loss_function = tf.keras.losses.Huber(
            reduction=tf.keras.losses.Reduction.NONE
        )

        self.gradient_steps = 0

    def build_model(
        self,
        hidden_layers,
        learning_rate
    ):
        """
        Build a neural network that predicts one Q-value
        for each discrete torque action.
        """

        model = tf.keras.Sequential()

        model.add(
            tf.keras.layers.Input(
                shape=(self.state_dimension,)
            )
        )

        for number_of_units in hidden_layers:
            model.add(
                tf.keras.layers.Dense(
                    number_of_units,
                    activation="relu"
                )
            )

        model.add(
            tf.keras.layers.Dense(
                self.number_of_actions,
                activation="linear"
            )
        )

        return model

    def select_action(self, state, epsilon):
        """
        Epsilon-greedy action selection.
        """

        # Exploration
        if np.random.random() < epsilon:
            return np.random.randint(
                self.number_of_actions
            )

        # Exploitation
        state_tensor = tf.convert_to_tensor(
            state.reshape(1, -1),
            dtype=tf.float32
        )

        q_values = self.online_model(
            state_tensor,
            training=False
        )

        return int(
            tf.argmax(q_values[0]).numpy()
        )

    @tf.function
    def _gradient_update(
        self,
        states,
        actions,
        rewards,
        next_states,
        terminals
    ):
        """
        Perform one Double DQN gradient update.

        Online model:
            Selects the best next action.

        Target model:
            Evaluates the selected next action.
        """

        # Online network chooses the next actions
        next_online_q_values = self.online_model(
            next_states,
            training=False
        )

        best_next_actions = tf.argmax(
            next_online_q_values,
            axis=1,
            output_type=tf.int32
        )

        # Target network evaluates those actions
        next_target_q_values = self.target_model(
            next_states,
            training=False
        )

        next_action_mask = tf.one_hot(
            best_next_actions,
            self.number_of_actions
        )

        selected_next_q_values = tf.reduce_sum(
            next_target_q_values * next_action_mask,
            axis=1
        )

        target_q_values = (
            rewards
            + self.gamma
            * (1.0 - terminals)
            * selected_next_q_values
        )

        target_q_values = tf.stop_gradient(
            target_q_values
        )

        with tf.GradientTape() as tape:
            current_q_values = self.online_model(
                states,
                training=True
            )

            current_action_mask = tf.one_hot(
                actions,
                self.number_of_actions
            )

            selected_current_q_values = tf.reduce_sum(
                current_q_values * current_action_mask,
                axis=1
            )

            loss_per_sample = self.loss_function(
                target_q_values,
                selected_current_q_values
            )

            loss = tf.reduce_mean(loss_per_sample)

        gradients = tape.gradient(
            loss,
            self.online_model.trainable_variables
        )

        gradients, _ = tf.clip_by_global_norm(
            gradients,
            10.0
        )

        self.optimizer.apply_gradients(
            zip(
                gradients,
                self.online_model.trainable_variables
            )
        )

        return loss

    def train_batch(self):
        """
        Sample a batch from replay memory and train the agent.
        """

        (
            states,
            actions,
            rewards,
            next_states,
            terminals
        ) = self.replay_buffer.sample(
            self.batch_size
        )

        states = tf.convert_to_tensor(
            states,
            dtype=tf.float32
        )

        actions = tf.convert_to_tensor(
            actions,
            dtype=tf.int32
        )

        rewards = tf.convert_to_tensor(
            rewards,
            dtype=tf.float32
        )

        next_states = tf.convert_to_tensor(
            next_states,
            dtype=tf.float32
        )

        terminals = tf.convert_to_tensor(
            terminals,
            dtype=tf.float32
        )

        loss = self._gradient_update(
            states,
            actions,
            rewards,
            next_states,
            terminals
        )

        self.gradient_steps += 1

        # Hard target-network update
        if (
            self.gradient_steps
            % self.target_update_train_steps
            == 0
        ):
            self.target_model.set_weights(
                self.online_model.get_weights()
            )

        return float(loss.numpy())

In [8]:
def set_all_seeds(seed):
    """
    Set Python, NumPy and TensorFlow random seeds.
    """
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def calculate_epsilon(
    current_step,
    epsilon_start,
    epsilon_end,
    epsilon_decay_steps
):
    """
    Linear epsilon decay.
    """

    progress = min(
        current_step / epsilon_decay_steps,
        1.0
    )

    epsilon = (
        epsilon_start
        + progress
        * (epsilon_end - epsilon_start)
    )

    return epsilon

In [9]:
def train_agent(
    gravity,
    config,
    seed,
    number_of_episodes,
    verbose=True
):
    """
    Train one Double DQN agent.

    Returns
    -------
    agent : DoubleDQNAgent
        Trained agent.

    history : pandas.DataFrame
        Episode rewards, losses, epsilon and upright percentage.
    """

    tf.keras.backend.clear_session()
    set_all_seeds(seed)

    env = create_pendulum_env(gravity)

    env.seed(seed)

    agent = DoubleDQNAgent(config)

    history_rows = []

    global_step = 0

    for episode in range(1, number_of_episodes + 1):
        raw_state = env.reset()
        state = preprocess_state(raw_state)

        done = False

        episode_reward = 0.0
        episode_losses = []

        upright_steps = 0
        episode_steps = 0

        while not done:
            epsilon = calculate_epsilon(
                current_step=global_step,
                epsilon_start=config["epsilon_start"],
                epsilon_end=config["epsilon_end"],
                epsilon_decay_steps=(
                    config["epsilon_decay_steps"]
                )
            )

            action_index = agent.select_action(
                state,
                epsilon
            )

            torque = agent.torque_values[
                action_index
            ]

            environment_action = np.array(
                [torque],
                dtype=np.float32
            )

            (
                next_raw_state,
                reward,
                done,
                info
            ) = env.step(environment_action)

            next_state = preprocess_state(
                next_raw_state
            )

            # Pendulum itself does not naturally terminate.
            # Gym's TimeLimit wrapper stops it after 200 steps.
            time_limit_reached = info.get(
                "TimeLimit.truncated",
                False
            )

            terminal_for_target = float(
                done and not time_limit_reached
            )

            agent.replay_buffer.add(
                state=state,
                action=action_index,
                reward=reward,
                next_state=next_state,
                terminal=terminal_for_target
            )

            # Calculate whether the pendulum is upright
            theta = np.arctan2(
                next_raw_state[1],
                next_raw_state[0]
            )

            angular_velocity = next_raw_state[2]

            is_upright = (
                abs(theta) <= UPRIGHT_ANGLE_LIMIT
                and
                abs(angular_velocity)
                <= UPRIGHT_SPEED_LIMIT
            )

            if is_upright:
                upright_steps += 1

            episode_steps += 1
            episode_reward += reward

            state = next_state
            global_step += 1

            # Begin training only after enough experience
            # has been collected.
            if (
                len(agent.replay_buffer)
                >= config["warmup_steps"]
                and
                global_step % config["train_every"] == 0
            ):
                loss = agent.train_batch()
                episode_losses.append(loss)

        mean_episode_loss = (
            np.mean(episode_losses)
            if len(episode_losses) > 0
            else np.nan
        )

        upright_percentage = (
            100.0
            * upright_steps
            / episode_steps
        )

        history_rows.append({
            "episode": episode,
            "reward": episode_reward,
            "loss": mean_episode_loss,
            "epsilon": epsilon,
            "upright_percentage": upright_percentage
        })

        if verbose and episode % 25 == 0:
            recent_rewards = [
                row["reward"]
                for row in history_rows[-25:]
            ]

            print(
                f"Gravity {gravity:>5} | "
                f"Seed {seed:>3} | "
                f"Episode {episode:>3}/{number_of_episodes} | "
                f"Mean reward {np.mean(recent_rewards):>8.2f} | "
                f"Epsilon {epsilon:.3f}"
            )

    env.close()

    history = pd.DataFrame(history_rows)

    return agent, history

In [10]:
def evaluate_agent(
    agent,
    gravity,
    evaluation_seeds
):
    """
    Evaluate a trained agent using unseen environment seeds.
    """

    env = create_pendulum_env(gravity)

    evaluation_rows = []

    for evaluation_seed in evaluation_seeds:
        env.seed(evaluation_seed)

        raw_state = env.reset()
        state = preprocess_state(raw_state)

        done = False

        total_reward = 0.0
        upright_steps = 0
        total_steps = 0
        total_absolute_torque = 0.0

        while not done:
            action_index = agent.select_action(
                state,
                epsilon=0.0
            )

            torque = agent.torque_values[
                action_index
            ]

            (
                next_raw_state,
                reward,
                done,
                info
            ) = env.step(
                np.array(
                    [torque],
                    dtype=np.float32
                )
            )

            theta = np.arctan2(
                next_raw_state[1],
                next_raw_state[0]
            )

            angular_velocity = next_raw_state[2]

            is_upright = (
                abs(theta) <= UPRIGHT_ANGLE_LIMIT
                and
                abs(angular_velocity)
                <= UPRIGHT_SPEED_LIMIT
            )

            if is_upright:
                upright_steps += 1

            total_reward += reward
            total_absolute_torque += abs(torque)
            total_steps += 1

            state = preprocess_state(
                next_raw_state
            )

        evaluation_rows.append({
            "evaluation_seed": evaluation_seed,
            "reward": total_reward,
            "upright_percentage": (
                100.0
                * upright_steps
                / total_steps
            ),
            "mean_absolute_torque": (
                total_absolute_torque
                / total_steps
            )
        })

    env.close()

    return pd.DataFrame(evaluation_rows)